In [ ]:
import os
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.layers import Conv2D, SeparableConv2D, MaxPool2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
import time

# ==========================================
# 1. TRAINING PIPELINE
# ==========================================

class PneumoniaModelTrainer:
    """Handles training, retraining, and metrics management for pneumonia detection model."""
    
    def __init__(self, data_path='./chest_xray/', img_dims=150, batch_size=32):
        self.data_path = data_path
        self.img_dims = img_dims
        self.batch_size = batch_size
        self.model = None
        self.metrics = {}
        self.metrics_file = 'model_metrics.json'
        
    def build_model(self):
        """Build CNN model architecture."""
        inputs = Input(shape=(self.img_dims, self.img_dims, 3))

        x = Conv2D(filters=16, kernel_size=(3, 3), activation='relu', padding='same')(inputs)
        x = Conv2D(filters=16, kernel_size=(3, 3), activation='relu', padding='same')(x)
        x = MaxPool2D(pool_size=(2, 2))(x)

        x = SeparableConv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same')(x)
        x = SeparableConv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same')(x)
        x = BatchNormalization()(x)
        x = MaxPool2D(pool_size=(2, 2))(x)

        x = SeparableConv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same')(x)
        x = SeparableConv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same')(x)
        x = BatchNormalization()(x)
        x = MaxPool2D(pool_size=(2, 2))(x)

        x = SeparableConv2D(filters=128, kernel_size=(3, 3), activation='relu', padding='same')(x)
        x = SeparableConv2D(filters=128, kernel_size=(3, 3), activation='relu', padding='same')(x)
        x = BatchNormalization()(x)
        x = MaxPool2D(pool_size=(2, 2))(x)
        x = Dropout(rate=0.2)(x)

        x = SeparableConv2D(filters=256, kernel_size=(3, 3), activation='relu', padding='same')(x)
        x = SeparableConv2D(filters=256, kernel_size=(3, 3), activation='relu', padding='same')(x)
        x = BatchNormalization()(x)
        x = MaxPool2D(pool_size=(2, 2))(x)
        x = Dropout(rate=0.2)(x)

        x = Flatten()(x)
        x = Dense(units=512, activation='relu')(x)
        x = Dropout(rate=0.7)(x)
        x = Dense(units=128, activation='relu')(x)
        x = Dropout(rate=0.5)(x)
        x = Dense(units=64, activation='relu')(x)
        x = Dropout(rate=0.3)(x)

        output = Dense(units=1, activation='sigmoid')(x)

        model = Model(inputs=inputs, outputs=output)
        return model
    
    def prepare_data(self):
        """Prepare training and validation data generators."""
        train_datagen = ImageDataGenerator(
            rescale=1./255,
            shear_range=0.2,
            zoom_range=0.2,
            horizontal_flip=True
        )
        
        test_val_datagen = ImageDataGenerator(rescale=1./255)
        
        train_gen = train_datagen.flow_from_directory(
            directory=os.path.join(self.data_path, 'train'),
            target_size=(self.img_dims, self.img_dims),
            batch_size=self.batch_size,
            class_mode='binary',
            shuffle=True
        )
        
        val_gen = test_val_datagen.flow_from_directory(
            directory=os.path.join(self.data_path, 'val'),
            target_size=(self.img_dims, self.img_dims),
            batch_size=self.batch_size,
            class_mode='binary',
            shuffle=False
        )
        
        return train_gen, val_gen
    
    def load_test_data(self):
        """Load and preprocess test data."""
        test_data = []
        test_labels = []
        
        for cond in ['NORMAL', 'PNEUMONIA']:
            label = 0 if cond == 'NORMAL' else 1
            cond_path = os.path.join(self.data_path, 'test', cond)
            
            for img_name in os.listdir(cond_path):
                img_path = os.path.join(cond_path, img_name)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                
                if img is None:
                    continue
                    
                img = cv2.resize(img, (self.img_dims, self.img_dims))
                img = np.dstack([img, img, img])
                img = img.astype('float32') / 255.0
                
                test_data.append(img)
                test_labels.append(label)
        
        return np.array(test_data), np.array(test_labels)
    
    def calculate_metrics(self, y_true, y_pred):
        """Calculate F1, Accuracy, Precision, and Recall."""
        y_pred_binary = np.round(y_pred)
        
        metrics = {
            'accuracy': float(accuracy_score(y_true, y_pred_binary)),
            'precision': float(precision_score(y_true, y_pred_binary)),
            'recall': float(recall_score(y_true, y_pred_binary)),
            'f1_score': float(f1_score(y_true, y_pred_binary)),
            'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
        }
        
        return metrics
    
    def save_metrics(self, metrics, weights_file='best_weights_pn.hdf5'):
        """Save metrics to JSON file."""
        metrics['weights_file'] = weights_file
        
        # Load existing metrics history or create new
        if os.path.exists(self.metrics_file):
            with open(self.metrics_file, 'r') as f:
                history = json.load(f)
        else:
            history = {'training_history': []}
        
        history['training_history'].append(metrics)
        history['latest'] = metrics
        
        with open(self.metrics_file, 'w') as f:
            json.dump(history, f, indent=4)
        
        print(f"✓ Metrics saved to {self.metrics_file}")
    
    def load_metrics(self):
        """Load metrics from JSON file."""
        if os.path.exists(self.metrics_file):
            with open(self.metrics_file, 'r') as f:
                return json.load(f)
        return None
    
    def train(self, epochs=10, weights_file='best_weights_pn.hdf5'):
        """Train the model and save best weights with metrics."""
        print("\n" + "="*60)
        print("TRAINING PIPELINE STARTED")
        print("="*60)
        
        # Build model
        self.model = self.build_model()
        self.model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
        
        print(f"✓ Model built with {len(self.model.layers)} layers")
        
        # Prepare data
        train_gen, val_gen = self.prepare_data()
        test_data, test_labels = self.load_test_data()
        
        print(f"✓ Data loaded - Train: {len(train_gen)}, Val: {len(val_gen)}, Test: {len(test_data)}")
        
        # Callbacks
        checkpoint = ModelCheckpoint(filepath=weights_file, save_best_only=True, save_weights_only=True)
        lr_reduce = ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, verbose=1)
        early_stop = EarlyStopping(monitor='val_loss', min_delta=0.1, patience=1, mode='min')
        
        # Train
        hist = self.model.fit(
            train_gen,
            steps_per_epoch=train_gen.samples // self.batch_size,
            epochs=epochs,
            validation_data=val_gen,
            validation_steps=val_gen.samples // self.batch_size,
            callbacks=[checkpoint, lr_reduce, early_stop]
        )
        
        # Load best weights
        self.model.load_weights(weights_file)
        
        # Calculate metrics on test data
        preds = self.model.predict(test_data, verbose=0)
        metrics = self.calculate_metrics(test_labels, preds)
        
        print("\n" + "-"*60)
        print("TEST METRICS")
        print("-"*60)
        print(f"Accuracy:  {metrics['accuracy']*100:.2f}%")
        print(f"Precision: {metrics['precision']*100:.2f}%")
        print(f"Recall:    {metrics['recall']*100:.2f}%")
        print(f"F1-Score:  {metrics['f1_score']:.4f}")
        print("-"*60)
        
        # Save metrics
        self.save_metrics(metrics, weights_file)
        
        print("\n" + "="*60)
        print(f"✓ TRAINING COMPLETE - Weights saved to {weights_file}")
        print("="*60 + "\n")
        
        return self.model, hist, metrics
    
    def retrain(self, original_weights='best_weights_pn.hdf5', epochs=5, learning_rate=1e-4):
        """Retrain the model from saved weights and create new weights file."""
        import os
        
        print("\n" + "="*60)
        print("RETRAINING PIPELINE STARTED")
        print("="*60)
        
        # Generate new weights filename with version
        base_name = os.path.splitext(original_weights)[0]
        ext = os.path.splitext(original_weights)[1]
        
        # Count existing retrained versions
        version = 1
        new_weights_file = f"{base_name}_v{version}{ext}"
        while os.path.exists(new_weights_file):
            version += 1
            new_weights_file = f"{base_name}_v{version}{ext}"
        
        # Load existing model and weights
        if self.model is None:
            self.model = self.build_model()
        
        self.model.load_weights(original_weights)
        print(f"✓ Loaded weights from {original_weights}")
        
        # Compile with new learning rate
        self.model.compile(optimizer=Adam(learning_rate=learning_rate), 
                          loss='binary_crossentropy', metrics=['accuracy'])
        
        # Prepare data
        train_gen, val_gen = self.prepare_data()
        test_data, test_labels = self.load_test_data()
        
        print(f"✓ Data loaded - Train: {len(train_gen)}, Val: {len(val_gen)}, Test: {len(test_data)}")
        print(f"✓ New weights will be saved to: {new_weights_file}")
        
        # Callbacks
        checkpoint = ModelCheckpoint(filepath=new_weights_file, save_best_only=True, save_weights_only=True)
        lr_reduce = ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=1, verbose=1)
        early_stop = EarlyStopping(monitor='val_loss', min_delta=0.05, patience=1, mode='min')
        
        # Retrain
        hist = self.model.fit(
            train_gen,
            steps_per_epoch=train_gen.samples // self.batch_size,
            epochs=epochs,
            validation_data=val_gen,
            validation_steps=val_gen.samples // self.batch_size,
            callbacks=[checkpoint, lr_reduce, early_stop]
        )
        
        # Load best weights
        self.model.load_weights(new_weights_file)
        
        # Calculate metrics on test data
        preds = self.model.predict(test_data, verbose=0)
        metrics = self.calculate_metrics(test_labels, preds)
        
        print("\n" + "-"*60)
        print("UPDATED TEST METRICS")
        print("-"*60)
        print(f"Accuracy:  {metrics['accuracy']*100:.2f}%")
        print(f"Precision: {metrics['precision']*100:.2f}%")
        print(f"Recall:    {metrics['recall']*100:.2f}%")
        print(f"F1-Score:  {metrics['f1_score']:.4f}")
        print("-"*60)
        
        # Save metrics
        self.save_metrics(metrics, new_weights_file)
        
        print("\n" + "="*60)
        print(f"✓ RETRAINING COMPLETE - New weights saved to {new_weights_file}")
        print("="*60 + "\n")
        
        return self.model, hist, metrics, new_weights_file


# ==========================================
# 2. TESTING PIPELINE
# ==========================================

class PneumoniaModelTester:
    """Handles inference and visualization for pneumonia detection model."""
    
    def __init__(self, model, img_dims=150, weights_path='best_weights_pn.hdf5'):
        self.model = model
        self.img_dims = img_dims
        self.weights_path = weights_path
        
    def predict_with_gradcam(self, image_path, show_visualization=True):
        """
        Predict on single image with Grad-CAM visualization.
        
        Args:
            image_path: Path to the image
            show_visualization: If True, display original and Grad-CAM side by side
            
        Returns:
            prob: Pneumonia probability (0-1)
            label: Predicted label ('NORMAL' or 'PNEUMONIA')
            overlay: Grad-CAM overlay image
        """
        self.model.load_weights(self.weights_path)
        
        # Load & preprocess image
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f'Image not found: {image_path}')
        
        orig = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        img_resized = cv2.resize(img, (self.img_dims, self.img_dims))
        x = np.stack([img_resized]*3, axis=-1).astype('float32') / 255.0
        x = np.expand_dims(x, axis=0)
        
        # Prediction
        prob = float(self.model.predict(x, verbose=0)[0][0])
        label = 'PNEUMONIA' if prob >= 0.5 else 'NORMAL'
        
        # Find last convolutional layer
        last_conv_layer = None
        for layer in reversed(self.model.layers):
            if isinstance(layer, tf.keras.layers.SeparableConv2D):
                last_conv_layer = layer.name
                break
        
        if last_conv_layer is None:
            raise RuntimeError("No SeparableConv2D layer found for Grad-CAM.")
        
        # Grad-CAM computation
        grad_model = tf.keras.models.Model(
            inputs=self.model.inputs,
            outputs=[self.model.get_layer(last_conv_layer).output, self.model.output]
        )
        
        with tf.GradientTape() as tape:
            conv_outputs, predictions = grad_model(x)
            loss = predictions[:, 0]
        
        grads = tape.gradient(loss, conv_outputs)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        
        conv_outputs = conv_outputs[0]
        heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
        
        # Normalize heatmap
        heatmap = tf.maximum(heatmap, 0)
        heatmap /= tf.reduce_max(heatmap) + 1e-8
        heatmap = heatmap.numpy()
        
        # Resize to original image
        heatmap = cv2.resize(heatmap, (orig.shape[1], orig.shape[0]))
        heatmap = np.uint8(255 * heatmap)
        
        heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
        overlay = cv2.addWeighted(orig, 0.6, heatmap_color, 0.4, 0)
        
        if show_visualization:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            
            # Original image on right
            axes[1].imshow(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB))
            axes[1].set_title('Original Image', fontsize=12, fontweight='bold')
            axes[1].axis('off')
            
            # Grad-CAM on left
            axes[0].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
            axes[0].set_title(f'Grad-CAM: {label} ({prob:.2%})', fontsize=12, fontweight='bold')
            axes[0].axis('off')
            
            plt.tight_layout()
            plt.show()
        
        return prob, label, overlay
    
    def test_folder(self, folder_path, output_dir='test_results'):
        """
        Test model on all images in a folder and save Grad-CAM results.
        
        Args:
            folder_path: Path to folder containing images
            output_dir: Directory to save results
        """
        os.makedirs(output_dir, exist_ok=True)
        
        self.model.load_weights(self.weights_path)
        
        valid_ext = (".png", ".jpg", ".jpeg", ".bmp")
        
        results = []
        pneumonia_count = 0
        normal_count = 0
        
        print("\n" + "="*60)
        print("TESTING FOLDER")
        print("="*60)
        print(f"Input folder: {folder_path}")
        print(f"Output folder: {output_dir}\n")
        
        for fname in sorted(os.listdir(folder_path)):
            if not fname.lower().endswith(valid_ext):
                continue
            
            img_path = os.path.join(folder_path, fname)
            
            try:
                prob, label, overlay = self.predict_with_gradcam(img_path, show_visualization=False)
                
                # Count predictions
                if label == 'PNEUMONIA':
                    pneumonia_count += 1
                else:
                    normal_count += 1
                
                # Save result
                prob_str = f"{prob:.2f}"
                base, ext = os.path.splitext(fname)
                out_name = f"{label}_{prob_str}_{base}{ext}"
                out_path = os.path.join(output_dir, out_name)
                
                cv2.imwrite(out_path, overlay)
                
                results.append({
                    'image': fname,
                    'prediction': label,
                    'probability': prob,
                    'output_file': out_name
                })
                
                print(f"✓ {fname:40s} -> {label:10s} ({prob:.2%})")
                
            except Exception as e:
                print(f"✗ {fname:40s} -> ERROR: {str(e)}")
        
        print("\n" + "-"*60)
        print("SUMMARY")
        print("-"*60)
        print(f"Total images tested: {len(results)}")
        print(f"Predicted PNEUMONIA: {pneumonia_count}")
        print(f"Predicted NORMAL:    {normal_count}")
        print(f"Results saved to:    {output_dir}")
        print("-"*60 + "\n")
        
        return results


In [ ]:
# ==========================================
# 3. INITIALIZATION
# ==========================================

# Initialize trainer and tester
trainer = PneumoniaModelTrainer(data_path='./chest_xray/', img_dims=150, batch_size=32)
pn_model = trainer.build_model()
tester = PneumoniaModelTester(pn_model, img_dims=150, weights_path='best_weights_pn.hdf5')

print("✓ Pneumonia Model Trainer and Tester initialized")
print("✓ Available commands:")
print("   - trainer.train(epochs=10)")
print("   - trainer.retrain(epochs=5)")
print("   - tester.predict_with_gradcam(image_path)")
print("   - tester.test_folder(folder_path)")

# Pneumonia Detection Model - Complete Training & Testing Pipeline

## Features:
- **Training Pipeline**: Train model from scratch with automatic metrics tracking
- **Retraining Pipeline**: Fine-tune existing model with updated metrics
- **Metrics Saved**: F1 Score, Accuracy, Precision, Recall stored in `model_metrics.json`
- **Single Image Testing**: Visualize predictions with Grad-CAM heatmaps
- **Batch Testing**: Test entire folders and save results with statistics

## Components:

### 1. PneumoniaModelTrainer
- `train()`: Initial model training
- `retrain()`: Fine-tune with existing weights
- `calculate_metrics()`: Compute F1, Accuracy, Precision, Recall
- `save_metrics()`: Persist metrics to JSON

### 2. PneumoniaModelTester
- `predict_with_gradcam()`: Single image prediction with visualization
- `test_folder()`: Batch testing with result aggregation

## Expected Data Structure

The training data should be organized as follows:

```
chest_xray/
├── train/
│   ├── NORMAL/
│   │   ├── image1.jpeg
│   │   ├── image2.jpeg
│   │   └── ... (more NORMAL images)
│   └── PNEUMONIA/
│       ├── image1.jpeg
│       ├── image2.jpeg
│       └── ... (more PNEUMONIA images)
├── val/
│   ├── NORMAL/
│   │   └── ... (validation NORMAL images)
│   └── PNEUMONIA/
│       └── ... (validation PNEUMONIA images)
└── test/
    ├── NORMAL/
    │   └── ... (test NORMAL images)
    └── PNEUMONIA/
        └── ... (test PNEUMONIA images)
```

### Requirements:
- **Image Format**: JPEG, PNG, BMP supported
- **Image Size**: Any size (automatically resized to 150x150)
- **Directory Structure**: Must have train/val/test subdirectories with NORMAL/PNEUMONIA classes
- **Minimum Images**: At least 100+ images per class recommended for good training

In [ ]:
# ==========================================
# TRAINING EXAMPLE
# ==========================================

# Step 1: Train from scratch
# model, history, metrics = trainer.train(epochs=10, weights_file='best_weights_pn.hdf5')

# This will:
# - Build the model
# - Load training/validation/test data
# - Train for 10 epochs with callbacks (checkpoint, LR reduction, early stopping)
# - Calculate metrics on test set
# - Save best weights to 'best_weights_pn.hdf5'
# - Save metrics to 'model_metrics.json' with F1, Accuracy, Precision, Recall

In [ ]:
# ==========================================
# RETRAINING EXAMPLE
# ==========================================

# Step 2: Retrain existing model (fine-tuning)
# model, history, metrics = trainer.retrain(
#     weights_file='best_weights_pn.hdf5',
#     epochs=5,
#     learning_rate=1e-4
# )

# This will:
# - Load existing weights
# - Retrain for 5 epochs with lower learning rate
# - Calculate updated metrics
# - Save improved weights
# - Update 'model_metrics.json' with new metrics

In [ ]:
# ==========================================
# TESTING ON SINGLE IMAGE
# ==========================================

# Test on single image with Grad-CAM visualization
# Shows original image on right, Grad-CAM heatmap on left
# prob, label, overlay = tester.predict_with_gradcam(
#     image_path='./chest_xray/test/PNEUMONIA/person11_virus_38.jpeg',
#     show_visualization=True
# )

# Returns:
# - prob: Probability of pneumonia (0-1)
# - label: Predicted label ('NORMAL' or 'PNEUMONIA')
# - overlay: Grad-CAM overlay image

In [ ]:
# ==========================================
# TESTING ON FOLDER
# ==========================================

# Test model on entire folder
# Saves Grad-CAM results in test_results/ folder
# results = tester.test_folder(
#     folder_path='./chest_xray/test/PNEUMONIA',
#     output_dir='test_results'
# )

# This will:
# - Process all images in the folder
# - Display predictions and probabilities
# - Save Grad-CAM overlays with format: {LABEL}_{PROBABILITY}_{ORIGINAL_NAME}
# - Print statistics: total count, pneumonia count, normal count
# - Return list of results with details

In [ ]:
# ==========================================
# VIEWING SAVED METRICS
# ==========================================

def view_metrics():
    """Display saved training metrics."""
    metrics = trainer.load_metrics()
    
    if not metrics:
        print("No metrics found. Run training first.")
        return
    
    print("\n" + "="*60)
    print("SAVED METRICS HISTORY")
    print("="*60)
    
    if 'latest' in metrics:
        latest = metrics['latest']
        print("\nLATEST METRICS:")
        print("-"*60)
        print(f"Timestamp:  {latest.get('timestamp', 'N/A')}")
        print(f"Weights:    {latest.get('weights_file', 'N/A')}")
        print(f"Accuracy:   {latest['accuracy']*100:.2f}%")
        print(f"Precision:  {latest['precision']*100:.2f}%")
        print(f"Recall:     {latest['recall']*100:.2f}%")
        print(f"F1-Score:   {latest['f1_score']:.4f}")
        print("-"*60)
    
    if 'training_history' in metrics:
        print(f"\nTRAINING HISTORY ({len(metrics['training_history'])} entries):")
        print("-"*60)
        for i, entry in enumerate(metrics['training_history'], 1):
            print(f"\n{i}. {entry.get('timestamp', 'N/A')} - {entry.get('weights_file', 'N/A')}")
            print(f"   Accuracy:  {entry['accuracy']*100:.2f}%")
            print(f"   Precision: {entry['precision']*100:.2f}%")
            print(f"   Recall:    {entry['recall']*100:.2f}%")
            print(f"   F1-Score:  {entry['f1_score']:.4f}")
    
    print("\n" + "="*60 + "\n")

# Uncomment to view metrics:
view_metrics()

## Complete Workflow Summary

### 1. Initial Training
```python
model, history, metrics = trainer.train(epochs=10, weights_file='best_weights_pn.hdf5')
```
- Trains from scratch
- Saves best weights to `best_weights_pn.hdf5`
- Records F1, Accuracy, Precision, Recall in `model_metrics.json`

### 2. Fine-tuning (Optional)
```python
model, history, metrics = trainer.retrain(epochs=5, learning_rate=1e-4)
```
- Loads existing weights
- Performs additional training
- Updates metrics in `model_metrics.json`
- Overwrites `best_weights_pn.hdf5` with new best weights

### 3. Single Image Testing
```python
prob, label, overlay = tester.predict_with_gradcam(
    image_path='./chest_xray/test/PNEUMONIA/person11_virus_38.jpeg'
)
```
- Loads weights from `best_weights_pn.hdf5` automatically
- Displays original image (right) and Grad-CAM heatmap (left)
- Returns prediction probability and label

### 4. Batch Testing
```python
results = tester.test_folder(
    folder_path='./chest_xray/test/PNEUMONIA',
    output_dir='test_results'
)
```
- Loads weights from `best_weights_pn.hdf5` automatically
- Tests all images in folder
- Saves Grad-CAM results in `test_results/` folder
- Prints statistics: total, pneumonia, normal counts

### 5. View Metrics
```python
view_metrics()
```
- Displays all saved metrics and training history

---

## Details on Weights Usage

### Automatic Weight Loading in Testing
The tester automatically uses the best weights saved during training:

```python
# During tester initialization
tester = PneumoniaModelTester(
    pn_model, 
    img_dims=150, 
    weights_path='best_weights_pn.hdf5'  # <-- Best weights file
)
```

### The Testing Pipeline:
1. **Before Prediction**: Loads best weights automatically
   ```python
   self.model.load_weights(self.weights_path)  # Loads 'best_weights_pn.hdf5'
   ```

2. **During Training**: Saves only the best epoch
   ```python
   checkpoint = ModelCheckpoint(
       filepath=weights_file,           # 'best_weights_pn.hdf5'
       save_best_only=True,             # Only save when validation improves
       save_weights_only=True
   )
   ```

3. **During Retraining**: Updates the same weights file
   ```python
   trainer.retrain(weights_file='best_weights_pn.hdf5')  # Updates existing best weights
   ```

### Verification
The metrics are always calculated on test data:
- **Training**: Metrics calculated after loading best weights
- **Retraining**: Metrics calculated after loading best weights from retraining
- **Testing**: Uses the same best weights file

This ensures consistency: the weights used for testing are the exact same weights that achieved the saved metrics.

In [ ]:
# ==========================================
# WEIGHTS & DATA VERIFICATION
# ==========================================

def verify_training_data_structure(data_path='./chest_xray/'):
    """Verify that training data has correct structure."""
    import os
    
    print("\n" + "="*70)
    print("TRAINING DATA STRUCTURE VERIFICATION")
    print("="*70)
    
    required_dirs = ['train', 'val', 'test']
    required_classes = ['NORMAL', 'PNEUMONIA']
    
    all_valid = True
    
    for dataset in required_dirs:
        dataset_path = os.path.join(data_path, dataset)
        print(f"\n{dataset.upper()} SET:")
        print("-"*70)
        
        if not os.path.exists(dataset_path):
            print(f"  ✗ Missing directory: {dataset_path}")
            all_valid = False
            continue
        
        for class_name in required_classes:
            class_path = os.path.join(dataset_path, class_name)
            
            if not os.path.exists(class_path):
                print(f"  ✗ Missing class directory: {class_path}")
                all_valid = False
            else:
                image_count = len([f for f in os.listdir(class_path) 
                                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
                if image_count > 0:
                    print(f"  ✓ {class_name:12s}: {image_count:4d} images")
                else:
                    print(f"  ✗ {class_name:12s}: No images found")
                    all_valid = False
    
    print("\n" + "="*70)
    if all_valid:
        print("✓ DATA STRUCTURE IS VALID")
    else:
        print("✗ DATA STRUCTURE HAS ISSUES - Please organize data correctly")
    print("="*70 + "\n")
    
    return all_valid


def verify_weights_and_metrics(weights_file='best_weights_pn.hdf5', metrics_file='model_metrics.json'):
    """Verify that best weights and metrics are available and synchronized."""
    import os
    import json
    
    print("\n" + "="*70)
    print("WEIGHTS & METRICS VERIFICATION")
    print("="*70)
    
    # Check weights file
    if os.path.exists(weights_file):
        file_size = os.path.getsize(weights_file) / (1024 * 1024)  # Convert to MB
        print(f"\n✓ Weights File: {weights_file}")
        print(f"  Size: {file_size:.2f} MB")
    else:
        print(f"\n✗ Weights File Not Found: {weights_file}")
        print("  → Run trainer.train() first to generate weights")
    
    # Check metrics file
    if os.path.exists(metrics_file):
        print(f"\n✓ Metrics File: {metrics_file}")
        with open(metrics_file, 'r') as f:
            metrics = json.load(f)
        
        if 'latest' in metrics:
            latest = metrics['latest']
            print(f"\n  Latest Training Session:")
            print(f"  - Timestamp:  {latest.get('timestamp', 'N/A')}")
            print(f"  - Weights:    {latest.get('weights_file', 'N/A')}")
            print(f"  - Accuracy:   {latest['accuracy']*100:.2f}%")
            print(f"  - Precision:  {latest['precision']*100:.2f}%")
            print(f"  - Recall:     {latest['recall']*100:.2f}%")
            print(f"  - F1-Score:   {latest['f1_score']:.4f}")
            
            # Verify synchronization
            if latest.get('weights_file') == weights_file:
                print(f"\n  ✓ SYNCHRONIZED: Metrics match weights file")
            else:
                print(f"\n  ⚠ WARNING: Metrics weights ({latest.get('weights_file')}) " + 
                      f"differ from tester weights ({weights_file})")
        
        if 'training_history' in metrics:
            history = metrics['training_history']
            print(f"\n  Training History: {len(history)} session(s)")
    else:
        print(f"\n✗ Metrics File Not Found: {metrics_file}")
        print("  → Run trainer.train() first to generate metrics")
    
    print("\n" + "="*70 + "\n")


def show_tester_weight_info():
    """Display current weights being used by tester."""
    print("\n" + "="*70)
    print("CURRENT TESTER CONFIGURATION")
    print("="*70)
    print(f"\nTester weights path: {tester.weights_path}")
    print(f"Tester image dims:   {tester.img_dims}x{tester.img_dims}")
    
    if os.path.exists(tester.weights_path):
        print(f"✓ Weights file exists and will be used for all predictions")
    else:
        print(f"✗ WARNING: Weights file does not exist!")
        print(f"  Run trainer.train() to generate: {tester.weights_path}")
    
    print("="*70 + "\n")


# ==========================================
# USAGE EXAMPLES
# ==========================================

# Verify data structure before training
# verify_training_data_structure('./chest_xray/')

# Verify weights and metrics after training
# verify_weights_and_metrics('best_weights_pn.hdf5', 'model_metrics.json')

# Show which weights the tester will use
# show_tester_weight_info()

## Practical Examples: Working with Weights

### Scenario 1: Fresh Start (Training + Testing)
```python
# Step 1: Train the model
model, history, metrics = trainer.train(epochs=10, weights_file='best_weights_pn.hdf5')

# Step 2: Current tester automatically uses these weights
# (weights_path='best_weights_pn.hdf5' set during initialization)
prob, label, overlay = tester.predict_with_gradcam('./chest_xray/test/PNEUMONIA/img.jpg')

# Verifies which weights are being used
show_tester_weight_info()
```

### Scenario 2: Using Different Weight Files
```python
# If you want to test with a different weights file:
tester_v2 = PneumoniaModelTester(
    model=pn_model,
    img_dims=150,
    weights_path='alternative_weights.hdf5'  # Different weights
)

# Now tester_v2 uses alternative_weights.hdf5
prob, label, overlay = tester_v2.predict_with_gradcam('./chest_xray/test/NORMAL/img.jpg')
```

### Scenario 3: Retraining and Testing with Updated Weights
```python
# Step 1: Retrain (updates best_weights_pn.hdf5)
model, history, metrics = trainer.retrain(epochs=5)

# Step 2: Tester automatically picks up updated weights
# Same tester object still uses best_weights_pn.hdf5
prob, label, overlay = tester.predict_with_gradcam('./chest_xray/test/PNEUMONIA/img.jpg')

# Verify new metrics are synchronized with weights
verify_weights_and_metrics('best_weights_pn.hdf5', 'model_metrics.json')
```

### Scenario 4: Batch Testing with Specific Weights
```python
# Test multiple folders ensuring best weights are used
results1 = tester.test_folder('./chest_xray/test/PNEUMONIA', 'results_pneumonia')
results2 = tester.test_folder('./chest_xray/test/NORMAL', 'results_normal')

# Both use the same best_weights_pn.hdf5 for consistency
```

### Scenario 5: Verify Data and Weights Before Operations
```python
# Always verify before training or testing
verify_training_data_structure('./chest_xray/')
verify_weights_and_metrics('best_weights_pn.hdf5', 'model_metrics.json')

# If everything is valid, proceed
if os.path.exists('best_weights_pn.hdf5'):
    show_tester_weight_info()
```

## Summary of Weights Flow

| Operation | Writes Weights | Reads Weights |
|-----------|---|---|
| **trainer.train()** | ✓ Saves to `best_weights_pn.hdf5` | ✓ Loads best epoch |
| **trainer.retrain()** | ✓ Updates `best_weights_pn.hdf5` | ✓ Loads best epoch |
| **tester.predict_with_gradcam()** | ✗ Read-only | ✓ Uses `weights_path` |
| **tester.test_folder()** | ✗ Read-only | ✓ Uses `weights_path` |
| **Metrics Calculation** | ✓ JSON file | ✓ Uses loaded best weights |

## Key Points
- ✓ **Training automatically saves best weights** to avoid overfitting
- ✓ **Testing automatically loads best weights** before prediction
- ✓ **Metrics are always synchronized** with weights used
- ✓ **Multiple testers can use different weights** if needed
- ✓ **Retraining updates the same weights file** for continuous improvement

---

## Step-by-Step Execution Guide

### Cell 1: Verify Data Structure
Run this first to ensure your data is properly organized

In [ ]:
# Cell 1: VERIFY DATA STRUCTURE
# ==============================
# Run this cell first to ensure your training data is properly organized
# Required directory structure: chest_xray/train/NORMAL, chest_xray/train/PNEUMONIA, etc.

verify_training_data_structure('./chest_xray/')
show_tester_weight_info()

### Cell 2: Train the Model (Initial Training)
This cell trains the model from scratch and saves weights to `best_weights_pn.hdf5`
- Saves best epoch automatically
- Records F1, Accuracy, Precision, Recall metrics
- Takes time depending on number of epochs

In [ ]:
# Cell 2: TRAIN THE MODEL (Initial Training)
# =============================================
# Trains model from scratch and saves to best_weights_pn.hdf5
# Duration: Depends on epochs and data size

# Configure training parameters
INITIAL_EPOCHS = 10

# Train the model
print("Starting initial training...")
model_trained, history_trained, metrics_trained = trainer.train(
    epochs=INITIAL_EPOCHS,
    weights_file='best_weights_pn.hdf5'
)

print("✓ Training complete! Weights saved to: best_weights_pn.hdf5")

### Cell 3: Retrain the Model (Fine-tuning)
This cell retrains from the best initial weights and saves to a **NEW** file (e.g., `best_weights_pn_v1.hdf5`)
- Creates new weights file automatically (versioning)
- Loads from previous best weights
- Uses lower learning rate for fine-tuning
- Preserves original weights file

In [ ]:
# Cell 3: RETRAIN THE MODEL (Fine-tuning)
# ========================================
# Retrains from best_weights_pn.hdf5 and saves to a NEW file (best_weights_pn_v1.hdf5, v2, etc.)
# Original weights remain unchanged for comparison

# Configure retraining parameters
RETRAIN_EPOCHS = 5
LEARNING_RATE = 1e-4

# Retrain the model
print("Starting retraining with fine-tuning...")
model_retrained, history_retrained, metrics_retrained, new_weights_file = trainer.retrain(
    original_weights='best_weights_pn.hdf5',
    epochs=RETRAIN_EPOCHS,
    learning_rate=LEARNING_RATE
)

print(f"✓ Retraining complete! New weights saved to: {new_weights_file}")
print(f"✓ Original weights preserved at: best_weights_pn.hdf5")

### Cell 4: Test the Model
Tests the model on images (single image or entire folder)
- **Single Image**: Shows original image (right) and Grad-CAM heatmap (left)
- **Folder**: Processes all images and saves Grad-CAM results with statistics

In [ ]:
# Cell 4A: TEST ON SINGLE IMAGE
# ==============================
# Tests on a single image with Grad-CAM visualization
# Shows predictions with heatmap overlay

image_test_path = './chest_xray/test/PNEUMONIA/person11_virus_38.jpeg'

print(f"Testing on: {image_test_path}\n")
prob, label, overlay = tester.predict_with_gradcam(
    image_path=image_test_path,
    show_visualization=True
)

print(f"\nPrediction: {label}")
print(f"Probability: {prob:.2%}")


In [ ]:
# ==============================
# Cell 4B: TEST ON ENTIRE FOLDER
# ==============================
# Uncomment below to test entire folder instead

test_folder_path = './chest_xray/test/PNEUMONIA'
results = tester.test_folder(
    folder_path=test_folder_path,
    output_dir='test_results'
)

print(f"\n✓ Testing complete!")
print(f"✓ Results saved to: test_results/")
print(f"✓ {len(results)} images processed")

### Cell 5: View Model Metrics
Display all saved metrics including:
- Latest metrics with timestamp
- Complete training history (all versions)
- Associated weights files
- Accuracy, Precision, Recall, F1-Score comparison

In [ ]:
# Cell 5: VIEW MODEL METRICS
# ==========================
# Display all training and retraining metrics
# Shows which weights file achieved which metrics

print("\n" + "="*70)
print("DISPLAYING ALL SAVED METRICS")
print("="*70)

view_metrics()

# Also verify current configuration
print("\nCurrent Tester Configuration:")
print(f"  Weights: {tester.weights_path}")
print(f"  Image size: {tester.img_dims}x{tester.img_dims}")

# Verify data structure
print("\nData Structure Status:")
verify_weights_and_metrics('best_weights_pn.hdf5', 'model_metrics.json')

---

## Summary: Workflow with Separate Weight Files

### Initial Training → Retraining Process

**Initial Training (Cell 2):**
- Trains from scratch
- Saves to: `best_weights_pn.hdf5`
- Saves metrics with this filename

**Retraining (Cell 3):**
- Loads from: `best_weights_pn.hdf5`
- Saves to: `best_weights_pn_v1.hdf5` (NEW file)
- Original weights remain unchanged
- Can retrain multiple times (creates v1, v2, v3, etc.)

**Testing (Cell 4):**
- By default uses: `best_weights_pn.hdf5`
- To test retrained model, change weights_path in tester
- Can compare predictions across different weight versions

**Metrics File (`model_metrics.json`):**
```json
{
  "training_history": [
    {
      "weights_file": "best_weights_pn.hdf5",
      "accuracy": 0.93,
      "precision": 0.92,
      "recall": 0.94,
      "f1_score": 0.9299,
      "timestamp": "2024-05-09 14:30:00"
    },
    {
      "weights_file": "best_weights_pn_v1.hdf5",
      "accuracy": 0.945,
      "precision": 0.935,
      "recall": 0.955,
      "f1_score": 0.945,
      "timestamp": "2024-05-09 15:45:00"
    }
  ],
  "latest": { ... }
}
```

### Benefits of This Approach:
✓ Compare multiple model versions  
✓ Preserve original training results  
✓ Track improvements over retraining  
✓ Easy rollback to previous versions  
✓ Complete audit trail in metrics file  

### To Test Different Weight Versions:
```python
# Create separate testers for each version
tester_v0 = PneumoniaModelTester(pn_model, weights_path='best_weights_pn.hdf5')
tester_v1 = PneumoniaModelTester(pn_model, weights_path='best_weights_pn_v1.hdf5')

# Compare predictions
prob_v0, label_v0, _ = tester_v0.predict_with_gradcam('image.jpg')
prob_v1, label_v1, _ = tester_v1.predict_with_gradcam('image.jpg')
```